In [ ]:
# Cell 1 - Imports
import os
import time
import csv
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from CLEAN.dataloader import *
from CLEAN.model import *
from CLEAN.utils import *
from CLEAN.distance_map import get_dist_map
from CLEAN.infer import *

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    average_precision_score,
    cohen_kappa_score,
    matthews_corrcoef,
)


In [ ]:
# Cell 2 - Default arguments
class Args:
    def __init__(self):
        self.learning_rate = 5e-4
        self.epoch = 20
        self.model_name = 'split100_triplet'
        self.training_data = 'split100'
        self.hidden_dim = 512
        self.out_dim = 128
        self.adaptive_rate = 100
        self.verbose = False
        self.batch_size = 6000
        self.hard_negative_topk = 30
        self.data_dir = './data'
        self.esm_data_dir = './data/esm_data'
        self.distance_map_dir = './data/distance_map'
        self.model_dir = './data/model'
        self.results_dir = './results'

args = Args()


In [ ]:
# Cell 3 - Helper functions
def normalize_ec(ec):
    ec = str(ec).strip()
    if ec.startswith('EC:'):
        ec = ec[3:]
    if '/' in ec:
        ec = ec.split('/', 1)[0]
    return ec


def pick_first_ec(ec_list_or_str):
    if isinstance(ec_list_or_str, (list, tuple, set)):
        if len(ec_list_or_str) == 0:
            return ''
        return normalize_ec(list(ec_list_or_str)[0])
    return normalize_ec(ec_list_or_str)


def random_positive2(seq_id, id_ec, ec_id):
    pos_ec = random.choice(list(id_ec[seq_id]))
    candidates = list(ec_id[pos_ec])

    if len(candidates) == 1:
        return candidates[0]

    valid = [x for x in candidates if x != seq_id]
    if not valid:
        return candidates[0]
    return random.choice(valid)


def calculate_ec_similarity(ec1_list, ec2_list):
    weights = [4, 3, 2, 1]
    similarities = []

    for ec1, ec2 in zip(ec1_list, ec2_list):
        ec1_levels = normalize_ec(ec1).split('.')
        ec2_levels = normalize_ec(ec2).split('.')

        max_levels = min(max(len(ec1_levels), len(ec2_levels)), len(weights))
        denominator = sum(weights[:max_levels]) if max_levels > 0 else 1

        similarity = 0.0
        for i in range(min(len(ec1_levels), len(ec2_levels), len(weights))):
            if ec1_levels[i] == ec2_levels[i]:
                similarity += weights[i]
            else:
                break

        similarities.append(similarity / denominator)

    return torch.tensor(similarities, dtype=torch.float32)


In [ ]:
# Cell 4 - Dataset and DataLoader
class Triplet_dataset_with_mine_EC2(torch.utils.data.Dataset):
    def __init__(self, id_ec, ec_id, mine_neg, esm_data_dir='./data/esm_data'):
        self.id_ec = id_ec
        self.ec_id = ec_id
        self.mine_neg = mine_neg
        self.esm_data_dir = esm_data_dir
        self.full_list = []

        for ec in ec_id.keys():
            if '-' not in str(ec):
                self.full_list.append(ec)

    def __len__(self):
        return len(self.full_list)

    def __getitem__(self, index):
        anchor_ec = self.full_list[index]
        anchor = random.choice(list(self.ec_id[anchor_ec]))

        pos = random_positive2(anchor, self.id_ec, self.ec_id)
        neg = mine_negative(anchor, self.id_ec, self.ec_id, self.mine_neg)

        a = torch.load(os.path.join(self.esm_data_dir, anchor + '.pt'), map_location='cpu')
        p = torch.load(os.path.join(self.esm_data_dir, pos + '.pt'), map_location='cpu')
        n = torch.load(os.path.join(self.esm_data_dir, neg + '.pt'), map_location='cpu')

        return (
            format_esm(a),
            format_esm(p),
            format_esm(n),
            self.id_ec[anchor],
            self.id_ec[neg],
            self.id_ec[pos],
        )


def collate_fn(batch):
    samples1, samples2, samples3, label1, label2, label3 = zip(*batch)
    return (samples1, samples2, samples3), label1, label2, label3


def get_dataloader2(dist_map, id_ec, ec_id, args):
    negative = mine_hard_negative(dist_map, args.hard_negative_topk)
    train_data = Triplet_dataset_with_mine_EC2(
        id_ec,
        ec_id,
        negative,
        esm_data_dir=args.esm_data_dir,
    )
    train_loader = torch.utils.data.DataLoader(
        train_data,
        batch_size=args.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
    )
    return train_loader


In [ ]:
# Cell 5 - Loss function
class TripletMarginLossWithEC(torch.nn.Module):
    def __init__(self, margin_max=1.0, margin_base=0.8):
        super(TripletMarginLossWithEC, self).__init__()
        self.margin_max = margin_max
        self.margin_base = margin_base

    def forward(self, anchor_embed, positive_embed, negative_embed, ec_sim_ap, ec_sim_an):
        ec_sim_ap = ec_sim_ap.view(-1)
        ec_sim_an = ec_sim_an.view(-1)
        ec_margin = self.margin_base + (self.margin_max - self.margin_base) * (ec_sim_ap - ec_sim_an)

        seq_loss = F.relu(
            torch.norm(anchor_embed - positive_embed, p=2, dim=1) ** 2
            - torch.norm(anchor_embed - negative_embed, p=2, dim=1) ** 2
            + ec_margin
        )
        return torch.mean(seq_loss)


In [ ]:
# Cell 6 - Training function
def train(model, args, epoch, train_loader, optimizer, device, dtype, criterion):
    model.train()
    total_loss = 0.0
    start_time = time.time()
    loss_values = []

    for batch, (samples, ec_a, ec_an, ec_ap) in enumerate(train_loader):
        optimizer.zero_grad()

        anchor, positive, negative = samples

        ec_anchor = [pick_first_ec(item) for item in ec_a]
        ec_positive = [pick_first_ec(item) for item in ec_ap]
        ec_negative = [pick_first_ec(item) for item in ec_an]

        anchor = torch.stack(anchor, dim=0).to(device=device, dtype=dtype)
        positive = torch.stack(positive, dim=0).to(device=device, dtype=dtype)
        negative = torch.stack(negative, dim=0).to(device=device, dtype=dtype)

        ec_sim_ap = calculate_ec_similarity(ec_anchor, ec_positive).to(device=device, dtype=dtype)
        ec_sim_an = calculate_ec_similarity(ec_anchor, ec_negative).to(device=device, dtype=dtype)

        anchor_out = model(anchor)
        positive_out = model(positive)
        negative_out = model(negative)

        loss = criterion(anchor_out, positive_out, negative_out, ec_sim_ap, ec_sim_an)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loss_values.append(loss.item())

        if args.verbose:
            lr = args.learning_rate
            ms_per_batch = (time.time() - start_time) * 1000
            cur_loss = total_loss
            print(f'| epoch {epoch:3d} | {batch:5d}/{len(train_loader):5d} batches | '
                  f'lr {lr:02.4f} | ms/batch {ms_per_batch:6.4f} | '
                  f'loss {cur_loss:5.2f}')
            start_time = time.time()

    return total_loss / (batch + 1), loss_values


In [7]:
# Cell 7 - Main training workflow
def main():
    seed_everything()
    ensure_dirs(args.model_dir)

    torch.backends.cudnn.benchmark = True
    id_ec, ec_id_dict = get_ec_id_dict(os.path.join(args.data_dir, args.training_data + '.csv'))
    ec_id = {key: list(ec_id_dict[key]) for key in ec_id_dict.keys()}

    use_cuda = torch.cuda.is_available()
    device = torch.device('cuda:0' if use_cuda else 'cpu')
    dtype = torch.float32
    lr, epochs = args.learning_rate, args.epoch
    model_name = args.model_name

    print('==> device used:', device, '| dtype used: ',
          dtype, "\n==> args:", args.__dict__)

    dist_map_path = os.path.join(args.distance_map_dir, args.training_data + '.pkl')
    dist_map = pickle.load(open(dist_map_path, 'rb'))

    model = LayerNormNet(args.hidden_dim, args.out_dim, device, dtype)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    criterion = TripletMarginLossWithEC()
    best_loss = float('inf')
    train_loader = get_dataloader2(dist_map, id_ec, ec_id, args)

    print('The number of unique EC numbers: ', len(dist_map.keys()))

    for epoch in range(1, epochs + 1):
        epoch_start_time = time.time()
        train_loss, loss_values = train(
            model,
            args,
            epoch,
            train_loader,
            optimizer,
            device,
            dtype,
            criterion,
        )

        if train_loss < best_loss and epoch > 0.8 * epochs:
            torch.save(model.state_dict(), os.path.join(args.model_dir, model_name + '.pth'))
            best_loss = train_loss
            print(f'Best from epoch : {epoch:3d}; loss: {train_loss:6.4f}')

        elapsed = time.time() - epoch_start_time
        print('-' * 75)
        print(f'| end of epoch {epoch:3d} | time: {elapsed:5.2f}s | '
              f'training loss {train_loss:6.4f}')
        print('-' * 75)

    torch.save(model.state_dict(), os.path.join(args.model_dir, model_name + '.pth'))
    return model


if __name__ == '__main__':
    main()


==> device used: cuda:0 | dtype used:  torch.float32 
==> args: {'learning_rate': 0.0005, 'epoch': 20, 'model_name': 'split100_triplet', 'training_data': 'split100', 'hidden_dim': 512, 'out_dim': 128, 'adaptive_rate': 100, 'verbose': False}
The number of unique EC numbers:  5242
---------------------------------------------------------------------------
| end of epoch   1 | time:  3.75s | training loss 1.2037
---------------------------------------------------------------------------
---------------------------------------------------------------------------
| end of epoch   2 | time:  2.81s | training loss 1.0522
---------------------------------------------------------------------------
---------------------------------------------------------------------------
| end of epoch   3 | time:  2.51s | training loss 0.9199
---------------------------------------------------------------------------
---------------------------------------------------------------------------
| end of epoch   

In [ ]:
# Cell 8 - Inference and evaluation utilities
def load_torch_checkpoint(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)


def write_max_sep_choices2(df, csv_name, first_grad=True, use_max_grad=True, gmm=None):
    gmm_lst = None
    if gmm is not None:
        gmm_lst = pickle.load(open(gmm, 'rb'))

    out_file = open(csv_name + '_maxsep.csv', 'w', newline='')
    csvwriter = csv.writer(out_file, delimiter=',')
    all_test_EC = set()

    for col in df.columns:
        ec = []
        smallest_10_dist_df = df[col].nsmallest(10)
        dist_lst = list(smallest_10_dist_df)
        max_sep_i = maximum_separation(dist_lst, first_grad, use_max_grad)
        max_sep_i = min(max_sep_i, len(smallest_10_dist_df) - 1)

        for i in range(max_sep_i + 1):
            EC_i = smallest_10_dist_df.index[i]
            dist_i = smallest_10_dist_df.iloc[i]

            if gmm_lst is not None:
                dist_i = infer_confidence_gmm(dist_i, gmm_lst)

            dist_str = '{:.4f}'.format(dist_i)
            all_test_EC.add(EC_i)
            ec.append('EC:' + str(EC_i) + '/' + dist_str)

        ec.insert(0, col)
        csvwriter.writerow(ec)

    out_file.close()
    return


def get_eval_metrics2(pred_label, pred_probs, true_label, all_label):
    mlb = MultiLabelBinarizer()
    mlb.fit([list(all_label)])

    n_test = len(pred_label)
    pred_m = np.zeros((n_test, len(mlb.classes_)))
    true_m = np.zeros((n_test, len(mlb.classes_)))
    pred_m_auc = np.zeros((n_test, len(mlb.classes_)))
    label_pos_dict = get_ec_pos_dict(mlb, true_label, pred_label)

    for i in range(n_test):
        pred_m[i] = mlb.transform([pred_label[i]])
        true_m[i] = mlb.transform([true_label[i]])
        labels, probs = pred_label[i], pred_probs[i]

        for label, prob in zip(labels, probs):
            if label in all_label:
                pos = label_pos_dict[label]
                pred_m_auc[i, pos] = prob

    pre = precision_score(true_m, pred_m, average='weighted', zero_division=0)
    rec = recall_score(true_m, pred_m, average='weighted', zero_division=0)
    f1 = f1_score(true_m, pred_m, average='weighted', zero_division=0)
    acc = accuracy_score(true_m, pred_m)

    try:
        roc_auc = roc_auc_score(true_m, pred_m_auc, average='weighted')
    except ValueError:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(true_m, pred_m_auc, average='weighted')
    except ValueError:
        pr_auc = np.nan

    conf_matrix = confusion_matrix(true_m.argmax(axis=1), pred_m.argmax(axis=1))
    kappa = cohen_kappa_score(true_m.argmax(axis=1), pred_m.argmax(axis=1))
    mcc = matthews_corrcoef(true_m.argmax(axis=1), pred_m.argmax(axis=1))

    return pre, rec, f1, acc, roc_auc, pr_auc, conf_matrix, kappa, mcc


In [9]:
# Cell 9 - Inference workflow
def infer_maxsep(train_data, test_data, report_metrics=False,
                 pretrained=True, model_name=None, gmm=None):
    use_cuda = torch.cuda.is_available()
    device = torch.device('cuda:0' if use_cuda else 'cpu')
    dtype = torch.float32

    id_ec_train, ec_id_dict_train = get_ec_id_dict(os.path.join(args.data_dir, train_data + '.csv'))
    id_ec_test, _ = get_ec_id_dict(os.path.join(args.data_dir, test_data + '.csv'))

    model = LayerNormNet(args.hidden_dim, args.out_dim, device, dtype)
    checkpoint = load_torch_checkpoint(os.path.join(args.model_dir, model_name + '.pth'), device)

    model.load_state_dict(checkpoint)
    model.eval()

    emb_train = model(esm_embedding(ec_id_dict_train, device, dtype))
    emb_test = model_embedding_test(id_ec_test, model, device, dtype)
    eval_dist = get_dist_map_test(emb_train, emb_test, ec_id_dict_train, id_ec_test, device, dtype)

    seed_everything()
    eval_df = pd.DataFrame.from_dict(eval_dist)
    ensure_dirs(args.results_dir)
    out_filename = os.path.join(args.results_dir, test_data)
    write_max_sep_choices2(eval_df, out_filename, gmm=gmm)

    if report_metrics:
        pred_label = get_pred_labels(out_filename, pred_type='_maxsep')
        pred_probs = get_pred_probs(out_filename, pred_type='_maxsep')
        true_label, all_label = get_true_labels(os.path.join(args.data_dir, test_data))

        pre, rec, f1, acc, roc_auc, pr_auc, conf_matrix, kappa, mcc = get_eval_metrics2(
            pred_label,
            pred_probs,
            true_label,
            all_label,
        )

        print('############ EC calling results using maximum separation ############')
        print('-' * 75)
        print(f'>>> total samples: {len(true_label)} | total ec: {len(all_label)} \n'
              f'>>> precision: {pre:.4} | recall: {rec:.4}'
              f'| F1: {f1:.4} | acc: {acc:.4} ')
        print(f'>>> ROC AUC: {roc_auc:.4} | PR AUC: {pr_auc:.4} ')
        print(f'>>> Confusion Matrix: \n{conf_matrix} ')
        print(f'>>> Kappa: {kappa:.4} | Matthews Correlation Coefficient: {mcc:.4} ')
        print('-' * 75)

    return eval_df


infer_maxsep('split100', 'new', report_metrics=True,
             pretrained=False, model_name='split100_triplet')


The embedding sizes for train and test: torch.Size([241025, 128]) torch.Size([392, 128])


100%|██████████| 5242/5242 [00:02<00:00, 1929.70it/s]


Calculating eval distance map, between 392 test ids and 5242 train EC cluster centers


392it [00:00, 644.83it/s]


############ EC calling results using maximum separation ############
---------------------------------------------------------------------------
>>> total samples: 392 | total ec: 177 
>>> precision: 0.5621 | recall: 0.4891| F1: 0.4829 | acc: 0.4362 
>>> ROC AUC: 0.7437 | PR AUC: 0.4655 
>>> Confusion Matrix: 
[[2 2 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 3 0]
 [1 0 0 ... 0 0 0]] 
>>> Kappa: 0.4762 | Matthews Correlation Coefficient: 0.5052 
---------------------------------------------------------------------------


In [ ]:
# Cell 10 - Original output record
# 100%|██████████| 5242/5242 [00:02<00:00, 1906.23it/s]
# Calculating eval distance map, between 392 test ids and 5242 train EC cluster centers
# 392it [00:00, 641.16it/s]
# ############ EC calling results using maximum separation ############
# ---------------------------------------------------------------------------
# >>> total samples: 392 | total ec: 177 
# >>> precision: 0.5483 | recall: 0.503| F1: 0.488 | acc: 0.4311 
# >>> ROC AUC: 0.7504 | PR AUC: 0.4676 
# >>> Confusion Matrix: 
# [[1 1 0 ... 0 0 0]
#  [0 1 0 ... 0 0 0]
#  [0 0 1 ... 0 0 0]
#  ...
#  [2 0 0 ... 0 0 0]
#  [0 0 0 ... 0 3 0]
#  [1 0 0 ... 0 0 0]] 
# >>> Kappa: 0.4615 | Matthews Correlation Coefficient: 0.4901 